# Korea Relative Valuation v1
## DuPont 2yr 예측 → PBR / PSR / PER 이론적 멀티플 → 적정주가 & 괴리율

### 핵심 공식 (US v4 동일)
```
PBR_theory = (ROE - g) / (Re - g)          [Gordon Growth]
PSR_theory = (NOPATm × (1-payout)) / (Re - g)
PER_theory = PBR_theory / ROE
Target Price = Multiple × Actual_Year2_Indicator
```

### ROE 구간별 g cap (v4 로직 유지)
| ROE 구간 | g_cap | 목적 |
|---|---|---|
| 저성장  ROE < 15% | `GDP_GROWTH` (2.5%) | 기업 성장률 한계 반영 |
| 중성장  15% ≤ ROE < 30% | `min(8%, Re×0.75)` | Spread ≥ Re×25% 보장 |
| 고성장  ROE ≥ 30%      | `Re × 0.75`        | 초과수익 지속가능성 반영 |

### US v4 → Korea v1 핵심 차이
| 항목 | US v4 | Korea v1 |
|---|---|---|
| 매출 데이터 | `us_revenue_forecast_data` (FMP) | `korea_fs_data_from_DG` + `korea_revenue_forecast_result` |
| 베타 | FMP `/beta` (fallback: 업종평균) | **자체 계산** (KSE_Price vs KOSPI 10년 회귀, Blume) |
| 섹터 | FMP `/profile` | FDR `StockListing('KRX')` (fallback: Unknown) |
| 현재가 | FMP `/quote` | `KSE_Price` 최신 close |
| 주식수 | FMP `weightedAverageShsOutDil` | DataGuide `S420004400` (자사주차감) |
| Payout | 배당 + Buyback | 배당 (`M001330710`)  ※ KR buyback 데이터 제한적 |
| GDP_GROWTH | 4% | **2.5%** (한국 추세) |
| Re 범위 | [7%, 18%] | [6%, 18%] |
| ERP | RF + 5.5% | **Damodaran 한국 7%** (또는 KOSPI geo 10y, floor 7%) |

### 저장
- 결과: `korea_relative_valuation` (PK: ticker, date)
- 데이터 품질: `korea_valuation_quality_log` (model='Relative')


## Cell 1 · 경로 자동 감지

In [1]:
import sys, os, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

_CANDIDATE_ROOTS = [
    r"C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast",
    r"C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy",
]

def _setup_path() -> str:
    try:
        start = Path(__file__).resolve().parent
    except NameError:
        start = Path.cwd()
    for p in [start] + list(start.parents):
        if (p / "DATA").is_dir():
            root = str(p)
            if root not in sys.path:
                sys.path.insert(0, root)
            print(f"[PATH] root 자동 감지 : {root}")
            return root
    for cand in _CANDIDATE_ROOTS:
        if os.path.isdir(cand) and os.path.isdir(os.path.join(cand, "DATA")):
            if cand not in sys.path:
                sys.path.insert(0, cand)
            print(f"[PATH] root 후보 경로 : {cand}")
            return cand
    raise EnvironmentError("DATA 폴더를 찾을 수 없습니다.")

_ROOT = _setup_path()
print(f"[확인] 프로젝트 루트 : {_ROOT}")


[PATH] root 자동 감지 : C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy
[확인] 프로젝트 루트 : C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy


## Cell 2 · Import & 설정 상수

> **조정 포인트**
> - `TICKER_START / TICKER_END / SKIP_DONE`
> - `ERP_METHOD` (Damodaran vs KOSPI geo)
> - `EXCLUDE_SECTORS` (FDR StockListing 기준 한국 섹터명)


In [2]:
import gc, math, time, traceback
from datetime import datetime, date, timedelta
from typing import Optional, Dict, Any, List, Tuple

import numpy as np
import pandas as pd
import pymysql
from scipy import stats
from IPython.display import display
import matplotlib
matplotlib.rcParams["axes.unicode_minus"] = False
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sqlalchemy import text

from DATA.config import get_db_info, get_engine
from DATA.KEYS import KEYS
from DATA.korea_valuation_helpers import (
    setup_project_path, to_dg_ticker, to_price_ticker, get_pymysql_conn,
    DG_ITEM_CODES,
    load_korea_financials_wide, load_korea_revenue_forecast,
    load_korea_marketcap_latest, load_korea_price_series, load_current_price,
    load_kospi_series, get_risk_free_rate, compute_beta_10y,
    estimate_market_return, get_universe_with_min_history,
    DataQualityReport, save_quality_report_to_db, get_evaluation_history,
)

def log(tag, msg):
    ts = datetime.now().strftime("%H:%M:%S")
    print(f"[{ts}][{tag}] {msg}", flush=True)

# ══════════════════════════════════════════════════════════════════
#  설정 상수 — 여기만 수정
# ══════════════════════════════════════════════════════════════════

# DB 테이블
TABLE_FS        = "korea_fs_data_from_DG"
TABLE_FORECAST  = "korea_revenue_forecast_result"
TABLE_PRICE     = "KSE_Price"
TABLE_MARKETCAP = "ks_listed_company_daily_marketcap"
TABLE_RESULT    = "korea_relative_valuation"
TABLE_QUALITY   = "korea_valuation_quality_log"

# 모델 파라미터 (US v4와 동일)
FORECAST_HORIZON    = 8
MIN_HISTORY         = 12
MIN_REVENUE_QUARTERS = 24
OLS_MIN_R2          = 0.20
OLS_MIN_SAMPLES     = 12
WINSORIZE_LIMITS    = (0.05, 0.95)

# 한국 GDP 성장률 (장기 추세)
GDP_GROWTH = 0.04

# ── ROE 구간별 g cap (US v4 로직) ──────────────────────────────
ROE_MID_THRESHOLD  = 0.15
ROE_HIGH_THRESHOLD = 0.30
G_CAP_MID_FIXED    = 0.08
G_CAP_RE_RATIO     = 0.75

# ── WACC / Re ──────────────────────────────────────────────────
ERP_METHOD        = "damodaran_floor"  # 'damodaran_floor' | 'kospi_geo_10y'
DAMODARAN_ERP_KR  = 0.07
GEO_FLOOR         = 0.07
RF_FALLBACK       = 0.035
RE_FLOOR          = 0.06   # 한국은 저금리 고려 6%
RE_CAP            = 0.18
BETA_DELTA        = 0.20   # Re 3종 격자 (low/mid/high)

# ── Beta 앙상블 ────────────────────────────────────────────────
W_INDIVIDUAL = 0.60
W_SECTOR     = 0.40

# ── 단위 변환 ──────────────────────────────────────────────────
FS_UNIT_MULTIPLIER        = 1_000        # 천원 → 원
MARKETCAP_UNIT_MULTIPLIER = 1_000_000    # 백만원 → 원 (확인 필요)

# ── 섹터 분류 (FDR StockListing 기준) ─────────────────────────
# KOSPI/KOSDAQ 상장사 섹터는 보통 한국어. FDR 결과 확인 후 조정 필요.
# 제외 업종: 금융/리츠/부동산/에너지 (회계 특성이 일반 기업과 달라 상대가치 부적절)
EXCLUDE_SECTORS = {
    "금융업", "은행", "증권", "보험",
    "부동산", "리츠", "REITs",
    "전기가스업",  # 공익사업
    "운수창고업",  # 해운/항공은 사이클성 강함
}

# 섹터별 평균 베타 (한국 시장 경험치, 필요 시 조정)
SECTOR_BETA = {
    "전기전자":     1.25, "IT":         1.30, "서비스업":   1.10,
    "화학":         1.10, "의약품":     0.85, "음식료업":   0.65,
    "유통업":       0.95, "운수장비":   1.15, "기계":       1.20,
    "철강금속":     1.25, "건설업":     1.30, "종이목재":   1.00,
    "섬유의복":     0.95, "비금속광물": 1.00, "의료정밀":   0.90,
    "통신업":       0.80, "제조업":     1.00,
    "Unknown":      1.00,
}

# ── 재무 항목 키 ───────────────────────────────────────────────
DEBT_KEYS = ["short_term_debt", "current_lt_debt", "bonds",
             "long_term_debt", "lease_liab"]
CASH_KEYS = ["cash", "short_term_invest"]

# ── 배치 범위 ──────────────────────────────────────────────────
TICKER_START = 0
TICKER_END   = 9999
SKIP_DONE    = False

CHECKPOINT_DIR = "_korea_relval_checkpoint"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
DONE_PATH = os.path.join(CHECKPOINT_DIR, "done_tickers.txt")
FAIL_PATH = os.path.join(CHECKPOINT_DIR, "failed_tickers.txt")

db_info = get_db_info()
engine  = get_engine(db_info)

print("[OK] Import 완료")
print(f"[설정] ERP_METHOD={ERP_METHOD}  GDP_GROWTH={GDP_GROWTH}")
print(f"[설정] Re 범위=[{RE_FLOOR:.0%}, {RE_CAP:.0%}]")
print(f"[설정] g_cap: 저성장=GDP({GDP_GROWTH:.1%}) | 중성장=min({G_CAP_MID_FIXED:.0%}, Re×{G_CAP_RE_RATIO})"
      f" | 고성장=Re×{G_CAP_RE_RATIO}")


[OK] Import 완료
[설정] ERP_METHOD=damodaran_floor  GDP_GROWTH=0.04
[설정] Re 범위=[6%, 18%]
[설정] g_cap: 저성장=GDP(4.0%) | 중성장=min(8%, Re×0.75) | 고성장=Re×0.75


## Cell 3 · DB 연결 & 결과 테이블 초기화

In [3]:
# ── 결과 테이블 생성 ──────────────────────────────────────────
CREATE_SQL = f"""
CREATE TABLE IF NOT EXISTS `{TABLE_RESULT}` (
  `id`            BIGINT      NOT NULL AUTO_INCREMENT,
  `date`          DATE        NOT NULL COMMENT '평가 실행일',
  `ticker`        VARCHAR(20) NOT NULL,
  `sector`        VARCHAR(50),
  `is_excluded`   TINYINT     DEFAULT 0,
  -- Sales 2yr
  `sales_y1`      DOUBLE,  `sales_y2`      DOUBLE,
  -- DuPont
  `npm_y1`        DOUBLE,  `npm_y2`        DOUBLE,  COMMENT_NPM_HINT VARCHAR(0) DEFAULT NULL,
  `at_y1`         DOUBLE,  `at_y2`         DOUBLE,
  `fl_y1`         DOUBLE,  `fl_y2`         DOUBLE,
  `roe_y1`        DOUBLE,  `roe_y2`        DOUBLE,
  `ni_y2`         DOUBLE,  `bve_y2`        DOUBLE,
  `eps_y2`        DOUBLE,  `bps_y2`        DOUBLE,  `sps_y2`        DOUBLE,
  `payout_ratio`  DOUBLE,  `g_est`         DOUBLE,
  -- Beta / Re
  `beta_raw`      DOUBLE,  `beta_blume`    DOUBLE,
  `beta_sector`   DOUBLE,  `beta_ensemble` DOUBLE,
  `re_low`        DOUBLE,  `re_mid`        DOUBLE,  `re_high`       DOUBLE,
  -- 이론 멀티플 & 적정가
  `pbr_theory`    DOUBLE,  `psr_theory`    DOUBLE,  `per_theory`    DOUBLE,
  `tp_pbr`        DOUBLE,  `tp_psr`        DOUBLE,  `tp_per`        DOUBLE,
  `tp_avg`        DOUBLE,
  `tp_pbr_low`    DOUBLE,  `tp_pbr_high`   DOUBLE,
  `tp_psr_low`    DOUBLE,  `tp_psr_high`   DOUBLE,
  `current_price` DOUBLE,
  `upside_pbr`    DOUBLE,  `upside_psr`    DOUBLE,
  `upside_per`    DOUBLE,  `upside_avg`    DOUBLE,
  `actual_pbr`    DOUBLE,  `actual_psr`    DOUBLE,  `actual_per` DOUBLE,
  `tax_rate`      DOUBLE,  `npm_r2`        DOUBLE,
  `created_at`    DATETIME DEFAULT CURRENT_TIMESTAMP,
  PRIMARY KEY (`id`),
  UNIQUE KEY uq_main (`ticker`, `date`),
  INDEX idx_ticker (`ticker`),
  INDEX idx_date   (`date`),
  INDEX idx_sector (`sector`)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4
"""

# ※ MySQL은 COMMENT 키워드가 컬럼 옵션으로 혼입되면 에러나므로 더미 표현 제거
CREATE_SQL = CREATE_SQL.replace("COMMENT_NPM_HINT VARCHAR(0) DEFAULT NULL,", "")

conn = get_pymysql_conn(db_info)
try:
    with conn.cursor() as cur:
        cur.execute(CREATE_SQL)
    conn.commit()
    log("DB", f"테이블 준비 완료: {TABLE_RESULT}")
finally:
    conn.close()

try:
    with engine.connect() as c:
        c.execute(text("SELECT 1"))
    log("DB", f"연결 성공 host={db_info.get('host')} port={db_info.get('port')}")
except Exception as e:
    log("DB", f"연결 실패: {e}")

# ── Universe 조회 ────────────────────────────────────────────
universe_df = get_universe_with_min_history(
    db_info, min_quarters=MIN_REVENUE_QUARTERS, require_consecutive=False)
KOREA_TICKER_LIST = universe_df["ticker"].tolist()
print(f"\n[Universe] 매출 ≥{MIN_REVENUE_QUARTERS}분기 종목: {len(KOREA_TICKER_LIST):,}개")


[00:08:12][DB] 테이블 준비 완료: korea_relative_valuation
[00:08:12][DB] 연결 성공 host=192.168.0.230 port=3307

[Universe] 매출 ≥24분기 종목: 1,267개


## Cell 4 · 시장 파라미터 + 한국 섹터 테이블 로드

- Rf, KOSPI, E(Rm): FCFF 노트북과 동일 로직
- `SECTOR_LOOKUP`: FDR `StockListing('KRX')` 로 한 번 로드 후 dict 캐싱
   - FDR 결과에 sector 컬럼이 없으면 모든 종목 'Unknown' → 섹터 베타 앙상블만 영향받음


In [4]:
# ── 1. 시장 파라미터 ─────────────────────────────────────────
RF, RF_SOURCE = get_risk_free_rate(KEYS["BOK"], fallback_rate=RF_FALLBACK)
log("MKT", f"Rf = {RF:.4%}  (source: {RF_SOURCE})")

KOSPI_PX = load_kospi_series(
    start_date=(datetime.today() - timedelta(days=365*12)).strftime("%Y-%m-%d"))
log("MKT", f"KOSPI {len(KOSPI_PX):,}거래일  "
            f"({KOSPI_PX.index.min().date()} ~ {KOSPI_PX.index.max().date()})")

mkt = estimate_market_return(
    method=ERP_METHOD, rf=RF, kospi_series=KOSPI_PX,
    years=10, damodaran_erp_kr=DAMODARAN_ERP_KR, geo_floor=GEO_FLOOR)
E_RM = mkt["e_rm"]
ERP  = mkt["erp"]
log("MKT", f"E(Rm) = {E_RM:.4%}  ERP = {ERP:.4%}  ({mkt['note']})")

# ── 2. 섹터 매핑 로드 (FDR StockListing) ──────────────────
SECTOR_LOOKUP: Dict[str, Dict[str, str]] = {}
try:
    import FinanceDataReader as fdr
    listing = fdr.StockListing("KRX")
    # 컬럼명은 FDR 버전에 따라 상이 — 유연 매칭
    code_col   = next((c for c in ["Code","Symbol","code","ticker"]
                       if c in listing.columns), None)
    sector_col = next((c for c in ["Sector","Industry","업종","sector","industry"]
                       if c in listing.columns), None)
    indu_col   = next((c for c in ["Industry","IndustryCode","업종명"]
                       if c in listing.columns and c != sector_col), None)
    name_col   = next((c for c in ["Name","name","기업명"]
                       if c in listing.columns), None)
    if code_col and sector_col:
        for _, row in listing.iterrows():
            code = str(row[code_col]).strip().zfill(6)
            SECTOR_LOOKUP[code] = {
                "sector":   str(row[sector_col]) if pd.notna(row[sector_col]) else "Unknown",
                "industry": str(row[indu_col])   if indu_col and pd.notna(row.get(indu_col)) else "",
                "name":     str(row[name_col])   if name_col and pd.notna(row.get(name_col)) else "",
            }
        log("SECTOR", f"FDR StockListing 로드 → {len(SECTOR_LOOKUP):,}개 종목 섹터 매핑")
        # 주요 섹터 개수 확인
        sector_cnts = pd.Series([v["sector"] for v in SECTOR_LOOKUP.values()]).value_counts()
        display(sector_cnts.head(15).to_frame("종목 수"))
    else:
        log("SECTOR", f"[WARN] FDR 컬럼명 불일치 (col={list(listing.columns)}) → 전체 Unknown")
except Exception as e:
    log("SECTOR", f"[WARN] FDR StockListing 실패: {e} → 전체 Unknown")


def get_sector(ticker) -> Dict[str, str]:
    """ticker → {'sector', 'industry', 'name'} (기본 Unknown)."""
    code = to_price_ticker(ticker)
    return SECTOR_LOOKUP.get(code, {"sector": "Unknown", "industry": "", "name": ""})


print()
print("=" * 60)
print(f"  Rf    = {RF:>7.3%}")
print(f"  ERP   = {ERP:>7.3%}")
print(f"  E(Rm) = {E_RM:>7.3%}")
print(f"  Sectors mapped: {len(SECTOR_LOOKUP):,}")
print("=" * 60)


[00:08:19][MKT] Rf = 4.0440%  (source: BOK_2026-05-13)
[KOSPI] try source=fdr (2014-05-17 ~ 2026-05-14) ... FAIL (FDR 실패 (재시도 3회): LOGOUT)
[KOSPI] try source=pykrx (2014-05-17 ~ 2026-05-14) ... FAIL ('지수명')
[KOSPI] try source=yfinance (2014-05-17 ~ 2026-05-14) ... OK  2,936거래일
[00:08:29][MKT] KOSPI 2,936거래일  (2014-05-19 ~ 2026-05-12)
[00:08:29][MKT] E(Rm) = 11.0440%  ERP = 7.0000%  (Damodaran 한국 ERP 7.0% 적용)
[00:08:29][SECTOR] [WARN] FDR StockListing 실패: Expecting value: line 1 column 1 (char 0) → 전체 Unknown

  Rf    =  4.044%
  ERP   =  7.000%
  E(Rm) = 11.044%
  Sectors mapped: 0


## Cell 5 · KoreaRelValModel 클래스

US v4 `RelativeValModel` 의 한국화. 로직은 동일, 데이터 소스만 변경.

In [5]:
# ═══════════════════════════════════════════════════════════════
#  KoreaRelValModel — DuPont 2yr → PBR/PSR/PER 상대가치
# ═══════════════════════════════════════════════════════════════

class KoreaRelValModel:
    """
    한국 주식 상대가치 3종 적정주가 평가.

    US v4 로직 그대로 사용 (Gordon Growth + ROE 구간별 g cap + NOPAT margin OLS).
    데이터 소스만 DataGuide DB 로 교체.
    """

    def __init__(self, ticker, engine, db_info, rf, e_rm, kospi_series,
                 verbose=False):
        self.ticker_dg    = to_dg_ticker(ticker)
        self.ticker_price = to_price_ticker(ticker)
        self.engine       = engine
        self.db_info      = db_info
        self.rf           = rf
        self.e_rm         = e_rm
        self.erp          = e_rm - rf
        self.kospi        = kospi_series
        self.verbose      = verbose

        self._sales_actual   = None
        self._sales_forecast = None
        self._used_model     = ""
        self._forecast_date  = None
        self._fs_wide        = None
        self._sector         = "Unknown"
        self._is_excluded    = False
        self.valuation       = None

        self.report = DataQualityReport(ticker=self.ticker_dg)

    @staticmethod
    def _winsorize(s):
        s = s.dropna()
        if len(s) < 4: return s
        lo, hi = s.quantile(WINSORIZE_LIMITS[0]), s.quantile(WINSORIZE_LIMITS[1])
        return s.clip(lo, hi)

    @staticmethod
    def _n(val):
        if val is None: return None
        try:
            f = float(val)
            return None if (np.isnan(f) or np.isinf(f)) else f
        except (TypeError, ValueError):
            return val

    # ─────────────────────────────────────────────────────────
    # 1. Sales 로드
    # ─────────────────────────────────────────────────────────
    def load_sales(self):
        wide = load_korea_financials_wide(
            self.ticker_dg, self.db_info, table_name=TABLE_FS,
            item_keys=["revenue"], fillna_zero=False)
        actual = wide["revenue"].dropna() * FS_UNIT_MULTIPLIER
        if actual.empty or len(actual) < MIN_REVENUE_QUARTERS:
            self.report.add("revenue_actual", "missing", n_obs=len(actual),
                            note=f"actual {len(actual)}Q < {MIN_REVENUE_QUARTERS}")
            raise ValueError(f"[{self.ticker_dg}] 매출 actual 부족 ({len(actual)}Q)")
        self.report.add("revenue_actual", "ok", n_obs=len(actual))

        forecast, model_name, ca = load_korea_revenue_forecast(
            self.ticker_dg, self.db_info, table_name=TABLE_FORECAST,
            horizon=FORECAST_HORIZON)
        if forecast.empty:
            self.report.add("revenue_forecast", "missing", n_obs=0,
                            note="korea_revenue_forecast_result 없음")
            raise ValueError(f"[{self.ticker_dg}] 매출 forecast 없음")
        forecast = forecast * FS_UNIT_MULTIPLIER
        self.report.add("revenue_forecast", "ok", n_obs=len(forecast),
                        note=f"model={model_name}")

        self._sales_actual   = actual
        self._sales_forecast = forecast.iloc[:FORECAST_HORIZON]
        self._used_model     = model_name
        self._forecast_date  = ca
        if self.verbose:
            log(self.ticker_dg,
                f"Sales actual={len(actual)}Q forecast={len(self._sales_forecast)}Q "
                f"({model_name})")
        return self

    # ─────────────────────────────────────────────────────────
    # 2. 재무제표 (DataGuide)
    # ─────────────────────────────────────────────────────────
    def load_financials(self):
        wide = load_korea_financials_wide(
            self.ticker_dg, self.db_info, table_name=TABLE_FS,
            item_keys=None, fillna_zero=False)
        if wide.empty:
            raise ValueError(f"[{self.ticker_dg}] FS 데이터 없음")

        non_share = [c for c in wide.columns
                     if c not in ("shares_treasury_adj", "shares_common")]
        wide[non_share] = wide[non_share] * FS_UNIT_MULTIPLIER

        # 핵심: 영업이익 필수
        if "operating_income" not in wide.columns or \
           wide["operating_income"].dropna().empty:
            self.report.add("operating_income", "missing", n_obs=0)
            raise ValueError(f"[{self.ticker_dg}] operating_income 없음")
        self.report.add("operating_income", "ok",
                        n_obs=int(wide["operating_income"].notna().sum()))

        self._fs_wide = wide
        if self.verbose:
            log(self.ticker_dg, f"FS wide shape={wide.shape}")
        return self

    # ─────────────────────────────────────────────────────────
    # 3. 섹터 로드
    # ─────────────────────────────────────────────────────────
    def load_sector(self):
        info = get_sector(self.ticker_dg)
        sector = info["sector"]
        self._sector      = sector
        self._is_excluded = sector in EXCLUDE_SECTORS
        self.report.add("sector", "ok" if sector != "Unknown" else "fallback_zero",
                        note=f"sector={sector}  excluded={self._is_excluded}")
        if self.verbose:
            excl = " [EXCLUDED]" if self._is_excluded else ""
            log(self.ticker_dg, f"sector={sector}{excl}")
        return self

    # ─────────────────────────────────────────────────────────
    # 4. Beta & Re (자체 계산)
    # ─────────────────────────────────────────────────────────
    def estimate_beta_re(self):
        """
        자체 베타 계산 (KSE_Price vs KOSPI 10년 회귀 + Blume) → 업종평균 앙상블.
        Re 3종 (low/mid/high) : β_ensemble ± BETA_DELTA → CAPM.
        """
        info = compute_beta_10y(
            self.ticker_dg, self.db_info, kospi_series=self.kospi,
            years=10, min_obs=750)

        beta_raw   = info["beta_raw"] if not np.isnan(info["beta_raw"]) else \
                     SECTOR_BETA.get(self._sector, 1.0)
        beta_blume = 0.67 * abs(beta_raw) + 0.33 if not np.isnan(beta_raw) else 1.0
        beta_sector = SECTOR_BETA.get(self._sector, 1.0)
        beta_ens    = W_INDIVIDUAL * beta_blume + W_SECTOR * beta_sector

        # fallback 기록
        if np.isnan(info["beta_raw"]):
            self.report.add("beta", "fallback_median", n_obs=info["n_obs"],
                            value=beta_sector,
                            note=f"자체 베타 실패 → 업종({self._sector}) 평균")
        else:
            self.report.add("beta", "ok", n_obs=info["n_obs"],
                            value=beta_blume, r2=info["r_squared"],
                            note=f"raw={info['beta_raw']:.3f}")

        def _re(b):
            return float(np.clip(self.rf + b * self.erp, RE_FLOOR, RE_CAP))
        re_mid  = _re(beta_ens)
        re_low  = _re(beta_ens - BETA_DELTA)
        re_high = _re(beta_ens + BETA_DELTA)

        result = {
            "beta_raw": float(beta_raw), "beta_blume": float(beta_blume),
            "beta_sector": float(beta_sector), "beta_ensemble": float(beta_ens),
            "re_low": re_low, "re_mid": re_mid, "re_high": re_high,
        }
        if self.verbose:
            log(self.ticker_dg,
                f"Beta raw={beta_raw:.3f} blume={beta_blume:.3f} "
                f"sector({self._sector})={beta_sector:.2f} ens={beta_ens:.3f} | "
                f"Re low={re_low:.3%} mid={re_mid:.3%} high={re_high:.3%}")
        return result

    # ─────────────────────────────────────────────────────────
    # 5. DuPont 2yr (NOPAT margin 기반, US v4 동일)
    # ─────────────────────────────────────────────────────────
    def estimate_dupont(self):
        wide = self._fs_wide

        # ── 실효세율 ────────────────────────────────────────
        tax = 0.22  # 한국 법정세율 fallback
        if "pretax_income" in wide.columns and "tax_expense" in wide.columns:
            df_tax = wide[["pretax_income", "tax_expense"]].dropna()
            df_tax = df_tax[df_tax["pretax_income"] > 0]
            if not df_tax.empty:
                rates = (df_tax["tax_expense"] / df_tax["pretax_income"]).clip(0, 0.40)
                tax = float(rates.median())
                self.report.add("tax_rate", "ok", n_obs=len(rates), value=tax)
            else:
                self.report.add("tax_rate", "fallback_zero", value=0.22,
                                note="pretax 양수 분기 없음 → 법정세율")
        else:
            self.report.add("tax_rate", "fallback_zero", value=0.22,
                            note="pretax/tax 컬럼 없음")

        # ── NOPAT margin OLS (revenue vs NOPAT = OI×(1-t)) ─
        if "revenue" in wide.columns and "operating_income" in wide.columns:
            df = wide[["revenue", "operating_income"]].dropna()
            df = df[df["revenue"] > 0].copy()
            df["nopat"] = df["operating_income"] * (1.0 - tax)

            nopat_series = self._winsorize((df["nopat"] / df["revenue"]).dropna())
            npm_median = float(nopat_series.median()) if not nopat_series.empty else 0.08
            npm_coef   = npm_median
            npm_r2     = -1.0

            if len(df) >= OLS_MIN_SAMPLES:
                slope, _, r, _, _ = stats.linregress(df["revenue"], df["nopat"])
                if r ** 2 >= OLS_MIN_R2 and slope >= 0:
                    npm_coef = float(slope)
                    npm_r2   = float(r ** 2)
                    self.report.add("nopat_margin", "ok", n_obs=len(df),
                                    value=npm_coef, r2=npm_r2, note="OLS slope")
                else:
                    self.report.add("nopat_margin", "fallback_median",
                                    n_obs=len(df), value=npm_median, r2=float(r**2),
                                    note=f"R²={r**2:.2f} < {OLS_MIN_R2} → median")
            else:
                self.report.add("nopat_margin", "fallback_median",
                                n_obs=len(df), value=npm_median,
                                note=f"n={len(df)} < {OLS_MIN_SAMPLES}")
        else:
            npm_coef, npm_median, npm_r2 = 0.08, 0.08, -1.0
            self.report.add("nopat_margin", "fallback_zero", value=0.08,
                            note="revenue/OI 컬럼 없음")

        # ── Asset Turnover (AT, TTM 기준) ───────────────────
        at_median = 0.70
        if "revenue" in wide.columns and "total_assets" in wide.columns:
            df_at = wide[["revenue", "total_assets"]].dropna().sort_index()
            df_at = df_at[df_at["total_assets"] > 0]
            if len(df_at) >= 4:
                df_at["ttm"] = df_at["revenue"].rolling(4).sum()
                atm = df_at.dropna(subset=["ttm"])
                at_ratios = self._winsorize((atm["ttm"] / atm["total_assets"]).dropna())
                if not at_ratios.empty:
                    at_median = float(at_ratios.median())
                    self.report.add("asset_turnover", "ok",
                                    n_obs=len(at_ratios), value=at_median)
            else:
                self.report.add("asset_turnover", "fallback_zero",
                                value=at_median, note="total_assets 4분기 미만")
        else:
            self.report.add("asset_turnover", "fallback_zero", value=at_median,
                            note="total_assets 컬럼 없음")

        # ── Financial Leverage (FL) ─────────────────────────
        fl_median = 2.5
        if "total_assets" in wide.columns and "total_equity" in wide.columns:
            df_fl = wide[["total_assets", "total_equity"]].dropna()
            df_fl = df_fl[(df_fl["total_assets"] > 0) & (df_fl["total_equity"] > 0)]
            if not df_fl.empty:
                fl_ratios = self._winsorize(
                    (df_fl["total_assets"] / df_fl["total_equity"]).dropna())
                fl_median = float(np.clip(fl_ratios.median(), 1.0, 20.0))
                self.report.add("financial_leverage", "ok",
                                n_obs=len(fl_ratios), value=fl_median)
            else:
                self.report.add("financial_leverage", "fallback_zero",
                                value=fl_median, note="자본/자산 양수 분기 없음")
        else:
            self.report.add("financial_leverage", "fallback_zero", value=fl_median)

        # ── Annual Sales 2yr ────────────────────────────────
        fc_q = self._sales_forecast
        annual_sales = []
        for yr in range(0, min(len(fc_q), 8), 4):
            annual_sales.append(float(fc_q.iloc[yr:yr+4].sum()))
        while len(annual_sales) < 2:
            annual_sales.append(annual_sales[-1] if annual_sales else 0)

        # ── BVE 시작값 ──────────────────────────────────────
        if "total_equity" in wide.columns:
            eq_ser = wide["total_equity"].dropna()
            bve0 = float(eq_ser.iloc[-1]) if not eq_ser.empty else 1e10
        else:
            bve0 = 1e10
        if bve0 <= 0: bve0 = 1e10

        # ── Shares (자사주차감 우선) ────────────────────────
        shares = np.nan
        for col in ["shares_treasury_adj", "shares_common"]:
            if col in wide.columns:
                s = wide[col].dropna()
                s = s[s > 0]
                if not s.empty:
                    shares = float(s.iloc[-1]); break
        if np.isnan(shares):
            self.report.add("shares", "missing", n_obs=0)
        else:
            self.report.add("shares", "ok", value=shares)

        # ── Payout ratio (배당만 사용; 한국 buyback 제한적) ─
        payout_med = 0.15  # 한국 기업 평균 배당성향
        if "dividends_paid" in wide.columns:
            div_ser = wide["dividends_paid"].abs().fillna(0)
            # NI 대신 NOPAT 기준 (v4 동일)
            nopat_ser = (wide["operating_income"] * (1.0 - tax)).abs()
            valid = (nopat_ser > 0) & (div_ser >= 0)
            if valid.sum() >= 4:
                po = (div_ser[valid] / nopat_ser[valid]).clip(0, 3.0)
                po_recent = po.iloc[-8:] if len(po) >= 8 else po
                payout_med = float(po_recent.median())
                self.report.add("payout_ratio", "ok", n_obs=int(valid.sum()),
                                value=payout_med, note="dividends_paid/NOPAT")
            else:
                self.report.add("payout_ratio", "fallback_zero",
                                value=payout_med, note=f"valid {valid.sum()} < 4")
        else:
            self.report.add("payout_ratio", "fallback_zero", value=payout_med,
                            note="dividends_paid 컬럼 없음")

        # ── 2yr DuPont 루프 ─────────────────────────────────
        results_yr = []
        bve = bve0
        for sales in annual_sales[:2]:
            nopat_est = npm_coef * sales
            npm       = nopat_est / sales if sales > 0 else 0.0
            roe       = float(np.clip(npm * at_median * fl_median, -0.99, 3.0))
            retention = max(1.0 - payout_med, -0.5)
            bve_end   = bve + nopat_est * retention
            results_yr.append({"sales": sales, "ni": nopat_est, "npm": npm,
                               "at": at_median, "fl": fl_median, "roe": roe,
                               "bve": bve_end})
            bve = bve_end

        y1, y2 = results_yr[0], results_yr[1]
        eps_y2 = y2["ni"] / shares  if (not np.isnan(shares) and shares > 0) else np.nan
        bps_y2 = y2["bve"]/ shares  if (not np.isnan(shares) and shares > 0) else np.nan
        sps_y2 = y2["sales"]/shares if (not np.isnan(shares) and shares > 0) else np.nan

        # ── g cap (v4 로직) ─────────────────────────────────
        _roe = y2["roe"]
        _g_raw = _roe * max(1.0 - payout_med, 0.0)
        if _roe >= ROE_HIGH_THRESHOLD:
            _g_cap_pre = G_CAP_MID_FIXED
        elif _roe >= ROE_MID_THRESHOLD:
            _g_cap_pre = G_CAP_MID_FIXED
        else:
            _g_cap_pre = GDP_GROWTH
        g_est = float(np.clip(_g_raw, 0.0, _g_cap_pre))

        if self.verbose:
            ols_tag = f"OLS(R²={npm_r2:.2f})" if npm_r2 >= OLS_MIN_R2 else "median"
            eps_s = f"{eps_y2:,.0f}원" if not np.isnan(eps_y2) else "N/A"
            bps_s = f"{bps_y2:,.0f}원" if not np.isnan(bps_y2) else "N/A"
            sps_s = f"{sps_y2:,.0f}원" if not np.isnan(sps_y2) else "N/A"
            log(self.ticker_dg,
                f"DuPont Y2 ROE={y2['roe']:.1%} NOPATm={y2['npm']:.1%}({ols_tag}) "
                f"AT={at_median:.2f} FL={fl_median:.1f} tax={tax:.1%} | "
                f"EPS={eps_s} BPS={bps_s} SPS={sps_s} g={g_est:.2%} "
                f"payout={payout_med:.1%}")

        return {
            "npm_y1": y1["npm"], "npm_y2": y2["npm"],
            "at_y1":  y1["at"],  "at_y2":  y2["at"],
            "fl_y1":  y1["fl"],  "fl_y2":  y2["fl"],
            "roe_y1": y1["roe"], "roe_y2": y2["roe"],
            "sales_y1": y1["sales"], "sales_y2": y2["sales"],
            "ni_y2":   y2["ni"],   "bve_y2": y2["bve"],
            "eps_y2":  eps_y2,     "bps_y2": bps_y2, "sps_y2": sps_y2,
            "payout_ratio": payout_med, "g_est": g_est,
            "shares": shares, "tax_rate": tax, "npm_r2": npm_r2,
        }

    # ─────────────────────────────────────────────────────────
    # 6. 이론 멀티플 & 적정가 (v4 로직 동일)
    # ─────────────────────────────────────────────────────────
    def compute_multiples(self, dupont, beta_re):
        roe = dupont["roe_y2"]; npm = dupont["npm_y2"]
        payout = dupont["payout_ratio"]
        eps = dupont["eps_y2"]; bps = dupont["bps_y2"]; sps = dupont["sps_y2"]
        re_mid  = beta_re["re_mid"]
        re_low  = beta_re["re_low"]
        re_high = beta_re["re_high"]

        def _calc_g(re_):
            retention = max(1.0 - payout, 0.0)
            g_raw = roe * retention
            if roe >= ROE_HIGH_THRESHOLD:
                g_cap = re_ * G_CAP_RE_RATIO
            elif roe >= ROE_MID_THRESHOLD:
                g_cap = min(G_CAP_MID_FIXED, re_ * G_CAP_RE_RATIO)
            else:
                g_cap = GDP_GROWTH
            return float(np.clip(g_raw, 0.0, g_cap))

        def _mult(re_):
            g = _calc_g(re_)
            sp = max(re_ - g, 0.005)
            pbr = float(np.clip((roe - g) / sp, 0.1, 100.0)) if roe > g else 0.1
            psr = float(np.clip(pbr * max(npm, 0) / roe, 0.01, 50.0)) if roe > 0 else np.nan
            per = float(np.clip(pbr / roe, 1.0, 200.0)) if roe > 0 else np.nan
            return pbr, psr, per, g

        pbr_m, psr_m, per_m, g_mid = _mult(re_mid)
        pbr_l, psr_l, _,     _     = _mult(re_low)
        pbr_h, psr_h, _,     _     = _mult(re_high)

        def _tp(mult, base):
            if np.isnan(base) or base <= 0 or np.isnan(mult): return np.nan
            return float(mult * base)

        tp_pbr = _tp(pbr_m, bps); tp_psr = _tp(psr_m, sps); tp_per = _tp(per_m, eps)
        tp_pbr_low = _tp(pbr_l, bps); tp_pbr_hi = _tp(pbr_h, bps)
        tp_psr_low = _tp(psr_l, sps); tp_psr_hi = _tp(psr_h, sps)

        valid = [x for x in [tp_pbr, tp_psr, tp_per] if not np.isnan(x) and x > 0]
        tp_avg = float(np.mean(valid)) if valid else np.nan

        # ── 현재가: KSE_Price 최신 ──────────────────────────
        cp = load_current_price(self.ticker_dg, self.db_info, table_name=TABLE_PRICE)
        if cp is None:
            self.report.add("current_price", "missing", n_obs=0)
            cp = np.nan

        def _up(tp):
            return (tp/cp - 1) * 100 if (not np.isnan(tp) and not np.isnan(cp) and cp > 0) else np.nan

        actual_pbr = float(cp / bps) if (not np.isnan(cp) and not np.isnan(bps) and bps > 0) else np.nan
        actual_psr = float(cp / sps) if (not np.isnan(cp) and not np.isnan(sps) and sps > 0) else np.nan
        actual_per = float(cp / eps) if (not np.isnan(cp) and not np.isnan(eps) and eps > 0) else np.nan

        if self.verbose:
            tp_str = f"{tp_avg:,.0f}원" if not np.isnan(tp_avg) else "N/A"
            cp_str = f"{cp:,.0f}원" if not np.isnan(cp) else "N/A"
            up_str = f"{_up(tp_avg):+.1f}%" if not np.isnan(_up(tp_avg)) else "N/A"
            log(self.ticker_dg,
                f"g={g_mid:.2%} PBR={pbr_m:.2f}x PSR={psr_m:.2f}x PER={per_m:.1f}x | "
                f"TP_avg={tp_str} CP={cp_str} Up={up_str}")

        return {
            "pbr_theory": pbr_m, "psr_theory": psr_m, "per_theory": per_m,
            "tp_pbr": tp_pbr, "tp_psr": tp_psr, "tp_per": tp_per, "tp_avg": tp_avg,
            "tp_pbr_low": tp_pbr_low, "tp_pbr_high": tp_pbr_hi,
            "tp_psr_low": tp_psr_low, "tp_psr_high": tp_psr_hi,
            "current_price": cp,
            "upside_pbr": _up(tp_pbr), "upside_psr": _up(tp_psr),
            "upside_per": _up(tp_per), "upside_avg": _up(tp_avg),
            "actual_pbr": actual_pbr, "actual_psr": actual_psr,
            "actual_per": actual_per,
        }

    # ─────────────────────────────────────────────────────────
    # 7. DB 저장
    # ─────────────────────────────────────────────────────────
    def save_to_db(self, run_date=None):
        if self.valuation is None: return False
        run_date = run_date or datetime.now().strftime("%Y-%m-%d")
        v = self.valuation; n = self._n
        row = {
            "date": run_date, "ticker": self.ticker_dg,
            "sector": self._sector, "is_excluded": 1 if self._is_excluded else 0,
            "sales_y1": n(v.get("sales_y1")), "sales_y2": n(v.get("sales_y2")),
            "npm_y1":   n(v.get("npm_y1")),   "npm_y2":   n(v.get("npm_y2")),
            "at_y1":    n(v.get("at_y1")),    "at_y2":    n(v.get("at_y2")),
            "fl_y1":    n(v.get("fl_y1")),    "fl_y2":    n(v.get("fl_y2")),
            "roe_y1":   n(v.get("roe_y1")),   "roe_y2":   n(v.get("roe_y2")),
            "ni_y2":    n(v.get("ni_y2")),    "bve_y2":   n(v.get("bve_y2")),
            "eps_y2":   n(v.get("eps_y2")),   "bps_y2":   n(v.get("bps_y2")),
            "sps_y2":   n(v.get("sps_y2")),
            "payout_ratio": n(v.get("payout_ratio")), "g_est": n(v.get("g_est")),
            "beta_raw":   n(v.get("beta_raw")),   "beta_blume":   n(v.get("beta_blume")),
            "beta_sector":n(v.get("beta_sector")),"beta_ensemble":n(v.get("beta_ensemble")),
            "re_low": n(v.get("re_low")), "re_mid": n(v.get("re_mid")),
            "re_high": n(v.get("re_high")),
            "pbr_theory": n(v.get("pbr_theory")), "psr_theory": n(v.get("psr_theory")),
            "per_theory": n(v.get("per_theory")),
            "tp_pbr": n(v.get("tp_pbr")), "tp_psr": n(v.get("tp_psr")),
            "tp_per": n(v.get("tp_per")), "tp_avg": n(v.get("tp_avg")),
            "tp_pbr_low": n(v.get("tp_pbr_low")), "tp_pbr_high": n(v.get("tp_pbr_high")),
            "tp_psr_low": n(v.get("tp_psr_low")), "tp_psr_high": n(v.get("tp_psr_high")),
            "current_price": n(v.get("current_price")),
            "upside_pbr": n(v.get("upside_pbr")), "upside_psr": n(v.get("upside_psr")),
            "upside_per": n(v.get("upside_per")), "upside_avg": n(v.get("upside_avg")),
            "actual_pbr": n(v.get("actual_pbr")), "actual_psr": n(v.get("actual_psr")),
            "actual_per": n(v.get("actual_per")),
            "tax_rate":   n(v.get("tax_rate")), "npm_r2": n(v.get("npm_r2")),
        }
        sql = f"""
        INSERT INTO `{TABLE_RESULT}`
        (date,ticker,sector,is_excluded,
         sales_y1,sales_y2,npm_y1,npm_y2,at_y1,at_y2,fl_y1,fl_y2,
         roe_y1,roe_y2,ni_y2,bve_y2,eps_y2,bps_y2,sps_y2,
         payout_ratio,g_est,
         beta_raw,beta_blume,beta_sector,beta_ensemble,
         re_low,re_mid,re_high,
         pbr_theory,psr_theory,per_theory,
         tp_pbr,tp_psr,tp_per,tp_avg,
         tp_pbr_low,tp_pbr_high,tp_psr_low,tp_psr_high,
         current_price,upside_pbr,upside_psr,upside_per,upside_avg,
         actual_pbr,actual_psr,actual_per,tax_rate,npm_r2)
        VALUES
        (%(date)s,%(ticker)s,%(sector)s,%(is_excluded)s,
         %(sales_y1)s,%(sales_y2)s,%(npm_y1)s,%(npm_y2)s,
         %(at_y1)s,%(at_y2)s,%(fl_y1)s,%(fl_y2)s,
         %(roe_y1)s,%(roe_y2)s,%(ni_y2)s,%(bve_y2)s,
         %(eps_y2)s,%(bps_y2)s,%(sps_y2)s,
         %(payout_ratio)s,%(g_est)s,
         %(beta_raw)s,%(beta_blume)s,%(beta_sector)s,%(beta_ensemble)s,
         %(re_low)s,%(re_mid)s,%(re_high)s,
         %(pbr_theory)s,%(psr_theory)s,%(per_theory)s,
         %(tp_pbr)s,%(tp_psr)s,%(tp_per)s,%(tp_avg)s,
         %(tp_pbr_low)s,%(tp_pbr_high)s,%(tp_psr_low)s,%(tp_psr_high)s,
         %(current_price)s,%(upside_pbr)s,%(upside_psr)s,%(upside_per)s,%(upside_avg)s,
         %(actual_pbr)s,%(actual_psr)s,%(actual_per)s,%(tax_rate)s,%(npm_r2)s)
        ON DUPLICATE KEY UPDATE
            tp_pbr=VALUES(tp_pbr), tp_psr=VALUES(tp_psr),
            tp_per=VALUES(tp_per), tp_avg=VALUES(tp_avg),
            upside_avg=VALUES(upside_avg), sector=VALUES(sector),
            roe_y2=VALUES(roe_y2), current_price=VALUES(current_price)
        """
        conn = get_pymysql_conn(self.db_info)
        try:
            with conn.cursor() as cur:
                cur.execute(sql, row)
            conn.commit()
            return True
        except Exception:
            conn.rollback(); raise
        finally:
            conn.close()

    # ─────────────────────────────────────────────────────────
    # 8. 전체 실행
    # ─────────────────────────────────────────────────────────
    def run(self):
        self.load_sales()
        self.load_financials()
        self.load_sector()
        beta_re = self.estimate_beta_re()
        dupont  = self.estimate_dupont()
        mults   = self.compute_multiples(dupont, beta_re)
        self.valuation = {**dupont, **beta_re, **mults}
        if self.verbose:
            v = self.valuation
            tp_s = f"{v['tp_avg']:,.0f}원" if not np.isnan(v['tp_avg']) else "N/A"
            up_s = f"{v['upside_avg']:+.1f}%" if not np.isnan(v['upside_avg']) else "N/A"
            log(self.ticker_dg,
                f"ROE_y2={v['roe_y2']:.1%} Re_mid={v['re_mid']:.3%} | "
                f"PBR={v['pbr_theory']:.2f}x PSR={v['psr_theory']:.2f}x | "
                f"TP_avg={tp_s} Up={up_s}")
        return self


def process_one_ticker_krv(ticker, engine, db_info, rf, e_rm, kospi_series,
                            verbose=False, run_date=None, save_db=True):
    run_date = run_date or datetime.now().strftime("%Y-%m-%d")
    try:
        m = KoreaRelValModel(ticker=ticker, engine=engine, db_info=db_info,
                             rf=rf, e_rm=e_rm, kospi_series=kospi_series,
                             verbose=verbose)
        m.run()
        saved = m.save_to_db(run_date) if save_db else False
        v = m.valuation
        return {
            "status": "ok", "ticker": m.ticker_dg,
            "sector": m._sector, "excluded": m._is_excluded,
            "tp_pbr": v.get("tp_pbr", np.nan), "tp_psr": v.get("tp_psr", np.nan),
            "tp_per": v.get("tp_per", np.nan), "tp_avg": v.get("tp_avg", np.nan),
            "upside_avg": v.get("upside_avg", np.nan),
            "roe_y2": v.get("roe_y2", np.nan), "re_mid": v.get("re_mid", np.nan),
            "saved": saved, "report": m.report,
        }
    except Exception as e:
        return {
            "status": "fail", "ticker": to_dg_ticker(ticker),
            "sector": "?", "excluded": False,
            "tp_pbr": np.nan, "tp_psr": np.nan, "tp_per": np.nan,
            "tp_avg": np.nan, "upside_avg": np.nan,
            "roe_y2": np.nan, "re_mid": np.nan,
            "saved": False, "report": None, "msg": str(e)[:150],
        }
    finally:
        gc.collect()


print("[OK] KoreaRelValModel 정의 완료")
print(f"     g_cap: 저성장 GDP({GDP_GROWTH:.1%}) | 중성장 min({G_CAP_MID_FIXED:.0%}, Re×{G_CAP_RE_RATIO}) "
      f"| 고성장 Re×{G_CAP_RE_RATIO}")

# ── 단일 티커 테스트 ─────────────────────────────────────────
TEST_TICKER = "A002900"
print(f"\n[테스트] {TEST_TICKER}")
_res = process_one_ticker_krv(TEST_TICKER, engine, db_info, RF, E_RM, KOSPI_PX,
                               verbose=True, save_db=False)
print(f"\n  status={_res['status']}  sector={_res['sector']}  excluded={_res['excluded']}")
if _res["status"] == "ok":
    tp_pbr_s = f"{_res['tp_pbr']:,.0f}원" if not np.isnan(_res['tp_pbr']) else "N/A"
    tp_psr_s = f"{_res['tp_psr']:,.0f}원" if not np.isnan(_res['tp_psr']) else "N/A"
    tp_avg_s = f"{_res['tp_avg']:,.0f}원" if not np.isnan(_res['tp_avg']) else "N/A"
    print(f"  TP_PBR={tp_pbr_s}  TP_PSR={tp_psr_s}  TP_avg={tp_avg_s}")
    print(f"  Upside_avg={_res['upside_avg']:+.1f}%")
    print()
    print("─" * 60); print("데이터 품질 리포트:"); print("─" * 60)
    print(_res["report"].summary())
    display(_res["report"].to_dataframe())


[OK] KoreaRelValModel 정의 완료
     g_cap: 저성장 GDP(4.0%) | 중성장 min(8%, Re×0.75) | 고성장 Re×0.75

[테스트] A002900
[00:08:29][A002900] Sales actual=65Q forecast=8Q (Ensemble)
[00:08:29][A002900] FS wide shape=(67, 35)
[00:08:29][A002900] sector=Unknown
[00:08:32][A002900] Beta raw=0.938 blume=0.959 sector(Unknown)=1.00 ens=0.975 | Re low=9.471% mid=10.871% high=12.271%
[00:08:32][A002900] DuPont Y2 ROE=22.4% NOPATm=8.8%(OLS(R²=0.55)) AT=1.02 FL=2.5 tax=24.2% | EPS=2,195원 BPS=14,594원 SPS=24,815원 g=8.00% payout=6.8%
[00:08:34][A002900] g=8.00% PBR=5.01x PSR=1.98x PER=22.4x | TP_avg=57,097원 CP=9,390원 Up=+508.1%
[00:08:34][A002900] ROE_y2=22.4% Re_mid=10.871% | PBR=5.01x PSR=1.98x | TP_avg=57,097원 Up=+508.1%

  status=ok  sector=Unknown  excluded=False
  TP_PBR=73,069원  TP_PSR=49,111원  TP_avg=57,097원
  Upside_avg=+508.1%

────────────────────────────────────────────────────────────
데이터 품질 리포트:
────────────────────────────────────────────────────────────
[A002900]  ok=10  fallback_median=0  fallback

,field,status,n_obs,value,r2,note
0,revenue_actual,ok,65,NaN,NaN,
1,revenue_forecast,ok,8,NaN,NaN,model=Ensemble
2,operating_income,ok,65,NaN,NaN,
3,sector,fallback_zero,0,NaN,NaN,sector=Unknown excluded=False
4,beta,ok,2449,9.587387e-01,0.095298,raw=0.938
5,tax_rate,ok,44,2.418279e-01,NaN,
6,nopat_margin,ok,65,8.843471e-02,0.552465,OLS slope
7,asset_turnover,ok,62,1.021397e+00,NaN,
8,financial_leverage,ok,65,2.476915e+00,NaN,
9,shares,ok,0,3.986104e+07,NaN,


## Cell 6 · 배치 실행

In [11]:
RUN_TICKERS = KOREA_TICKER_LIST[TICKER_START:TICKER_END]
total = len(RUN_TICKERS)
run_date = datetime.now().strftime("%Y-%m-%d")

done_set = set()
if SKIP_DONE and os.path.exists(DONE_PATH):
    with open(DONE_PATH, encoding="utf-8") as f:
        done_set = {l.strip() for l in f if l.strip()}

ok_cnt = skip_cnt = fail_cnt = excl_cnt = 0
results, all_reports = [], []
t0 = time.time()

log("BATCH", f"상대가치 배치 시작: {total:,}개  run_date={run_date}")
print("=" * 80)

for idx, ticker in enumerate(RUN_TICKERS, 1):
    pct = idx / total * 100
    prefix = f"[{idx:>5}/{total}] ({pct:5.1f}%) {ticker:<8}"

    if SKIP_DONE and ticker in done_set:
        print(f"{prefix} SKIP", flush=True); skip_cnt += 1; continue

    res = process_one_ticker_krv(
        ticker, engine, db_info, RF, E_RM, KOSPI_PX,
        verbose=False, run_date=run_date, save_db=True)
    results.append(res)
    if res["report"] is not None:
        all_reports.append(res["report"])

    if res["status"] == "ok":
        tp_s = f"TP={res['tp_avg']:,.0f}원" if not np.isnan(res["tp_avg"]) else "TP=N/A"
        up_s = f"↑{res['upside_avg']:+.1f}%" if not np.isnan(res["upside_avg"]) else ""
        excl_s = " [EXCL]" if res["excluded"] else ""
        print(f"{prefix} OK  {tp_s} {up_s}  {res['sector']}{excl_s}", flush=True)
        with open(DONE_PATH, "a", encoding="utf-8") as f:
            f.write(ticker + "\n")
        ok_cnt += 1
        if res["excluded"]: excl_cnt += 1
    else:
        print(f"{prefix} FAIL  {res.get('msg','')}", flush=True)
        with open(FAIL_PATH, "a", encoding="utf-8") as f:
            f.write(f"{ticker}\t{res.get('msg','')}\n")
        fail_cnt += 1

elapsed = time.time() - t0
print("=" * 80)
log("BATCH", f"완료  OK={ok_cnt}  EXCL={excl_cnt}  FAIL={fail_cnt}  "
             f"경과={elapsed:.0f}s  평균={elapsed/max(ok_cnt+fail_cnt,1):.1f}s/ticker")

# 데이터 품질 로그 저장
if all_reports:
    n_q = save_quality_report_to_db(
        all_reports, db_info, table_name=TABLE_QUALITY,
        run_date=run_date, model_name="Relative")
    log("QUALITY", f"품질 로그 {n_q:,}건 → {TABLE_QUALITY}")

# 요약
if results:
    summary = pd.DataFrame([{
        "ticker": r["ticker"], "sector": r["sector"], "excluded": r["excluded"],
        "roe_y2": r["roe_y2"], "re_mid": r["re_mid"],
        "tp_avg": r["tp_avg"], "upside_avg": r["upside_avg"],
        "status": r["status"],
    } for r in results])
    ok_summary = summary[(summary["status"] == "ok") & (~summary["excluded"])]
    ok_summary = ok_summary.sort_values("upside_avg", ascending=False)
    print("\n[상위 업사이드 — 비제외 종목 TOP 20]")
    display(ok_summary.head(20))


[22:23:14][BATCH] 상대가치 배치 시작: 1,267개  run_date=2026-05-02


KeyboardInterrupt: 

## Cell 7 · 결과 조회 & 시각화

In [ ]:
# 최신 평가일 기준 결과 조회
conn = get_pymysql_conn(db_info)
try:
    with conn.cursor() as cur:
        cur.execute(f"SELECT MAX(date) AS m FROM {TABLE_RESULT}")
        max_date = cur.fetchone()["m"]
        if max_date:
            cur.execute(f"""
                SELECT ticker, sector, is_excluded,
                       roe_y2, re_mid, g_est,
                       pbr_theory, psr_theory, per_theory,
                       tp_pbr, tp_psr, tp_per, tp_avg,
                       current_price, upside_avg,
                       actual_pbr, actual_psr, actual_per,
                       npm_r2
                FROM {TABLE_RESULT}
                WHERE date = %s AND is_excluded = 0
                  AND tp_avg IS NOT NULL AND current_price > 0
                ORDER BY upside_avg DESC
            """, (max_date,))
            results_df = pd.DataFrame(cur.fetchall())
        else:
            results_df = pd.DataFrame()
finally:
    conn.close()

if not results_df.empty:
    print(f"[최신 평가일: {max_date}] 비제외 종목 {len(results_df):,}개")
    print()
    print("🔼 업사이드 TOP 20")
    display(results_df.head(20))
    print()
    print("🔽 다운사이드 TOP 20")
    display(results_df.tail(20))

    # 분포 + 섹터별 평균
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f"Korea Relative Valuation Summary ({max_date})", fontsize=13)

    ax = axes[0]
    upside_clean = results_df["upside_avg"].clip(-200, 200).dropna()
    ax.hist(upside_clean, bins=40, color="#3498db", edgecolor="white", alpha=0.8)
    ax.axvline(0, color="black", lw=1)
    ax.axvline(20, color="green", lw=1, ls="--", label="+20%")
    ax.set_title("Upside Distribution (%)")
    ax.set_xlabel("Upside"); ax.legend(); ax.grid(alpha=0.3)

    ax = axes[1]
    grade = pd.cut(results_df["upside_avg"],
                   bins=[-np.inf, -20, 0, 20, 50, np.inf],
                   labels=["Strong Sell", "Sell", "Hold", "Buy", "Strong Buy"])
    grade.value_counts().sort_index().plot(
        kind="barh", ax=ax,
        color=["#c0392b", "#e74c3c", "#f39c12", "#2ecc71", "#27ae60"])
    ax.set_title("Valuation Grade"); ax.grid(axis="x", alpha=0.3)

    ax = axes[2]
    sec_stats = (results_df.groupby("sector")["upside_avg"]
                  .agg(["mean", "count"]).sort_values("mean", ascending=False))
    sec_stats = sec_stats[sec_stats["count"] >= 3].head(15)
    ax.barh(range(len(sec_stats)), sec_stats["mean"],
            color="#2980b9", alpha=0.8)
    ax.set_yticks(range(len(sec_stats)))
    ax.set_yticklabels([f"{s} (n={int(c)})" for s, c in
                        zip(sec_stats.index, sec_stats["count"])], fontsize=8)
    ax.axvline(0, color="black", lw=1)
    ax.set_title("Avg Upside by Sector (n≥3)"); ax.grid(axis="x", alpha=0.3)

    plt.tight_layout(); plt.show()
else:
    print("[INFO] 데이터 없음 — Cell 6 배치 실행 먼저")


## Cell 8 · 데이터 품질 진단

호영님 결정 #5. 배치 실행 시 모든 fallback 내역이 `korea_valuation_quality_log` 에 저장됨.
여기서 필터링해서 어떤 종목의 어떤 항목이 문제였는지 확인 가능.


In [ ]:
# ── 데이터 품질 진단 (Relative 모델) ────────────────────
conn = get_pymysql_conn(db_info)
try:
    with conn.cursor() as cur:
        cur.execute(f"""
            SELECT field, status, COUNT(*) AS cnt,
                   AVG(r_squared) AS avg_r2, AVG(value) AS avg_value
            FROM {TABLE_QUALITY}
            WHERE model='Relative' AND date=(SELECT MAX(date) FROM {TABLE_QUALITY}
                                              WHERE model='Relative')
            GROUP BY field, status
            ORDER BY field, status
        """)
        df_status = pd.DataFrame(cur.fetchall())

        cur.execute(f"""
            SELECT field,
                   SUM(status='ok') AS ok_cnt,
                   SUM(status='fallback_median') AS fb_med,
                   SUM(status='fallback_zero') AS fb_zero,
                   SUM(status='missing') AS miss,
                   COUNT(*) AS total,
                   ROUND(SUM(status LIKE 'fallback%')/COUNT(*)*100, 1) AS fb_pct
            FROM {TABLE_QUALITY}
            WHERE model='Relative' AND date=(SELECT MAX(date) FROM {TABLE_QUALITY}
                                              WHERE model='Relative')
            GROUP BY field
            ORDER BY fb_pct DESC
        """)
        df_field = pd.DataFrame(cur.fetchall())

        cur.execute(f"""
            SELECT ticker,
                   SUM(status LIKE 'fallback%') AS fallback_cnt,
                   SUM(status='missing') AS miss_cnt,
                   GROUP_CONCAT(DISTINCT field ORDER BY field SEPARATOR ',') AS issues
            FROM {TABLE_QUALITY}
            WHERE model='Relative' AND date=(SELECT MAX(date) FROM {TABLE_QUALITY}
                                              WHERE model='Relative')
              AND (status LIKE 'fallback%' OR status='missing')
            GROUP BY ticker
            HAVING fallback_cnt + miss_cnt >= 3
            ORDER BY (fallback_cnt + miss_cnt*2) DESC
            LIMIT 30
        """)
        df_problem = pd.DataFrame(cur.fetchall())
finally:
    conn.close()

print("=" * 70); print("[1] 항목별 status 분포"); print("=" * 70); display(df_status)
print("\n" + "=" * 70); print("[2] 항목별 fallback 비율"); print("=" * 70); display(df_field)
print("\n" + "=" * 70); print("[3] 문제 많은 종목 TOP 30"); print("=" * 70); display(df_problem)


def inspect_rel_quality(ticker: str, run_date: str = None) -> pd.DataFrame:
    """특정 종목의 Relative 모델 데이터 품질 상세."""
    tk = to_dg_ticker(ticker)
    conn = get_pymysql_conn(db_info)
    try:
        with conn.cursor() as cur:
            if run_date:
                cur.execute(f"""SELECT date, field, status, n_obs, value, r_squared, note
                                  FROM {TABLE_QUALITY}
                                  WHERE ticker=%s AND date=%s AND model='Relative'
                                  ORDER BY field""", (tk, run_date))
            else:
                cur.execute(f"""SELECT date, field, status, n_obs, value, r_squared, note
                                  FROM {TABLE_QUALITY}
                                  WHERE ticker=%s AND model='Relative'
                                    AND date=(SELECT MAX(date) FROM {TABLE_QUALITY}
                                               WHERE ticker=%s AND model='Relative')
                                  ORDER BY field""", (tk, tk))
            return pd.DataFrame(cur.fetchall())
    finally:
        conn.close()

print("\n[예시] 삼성전자 Relative 품질 상세")
display(inspect_rel_quality("A005930"))


## Cell 9 · 종목별 평가 이력 조회

특정 종목의 날짜별 상대가치 평가 추이. (호영님 요청 — 평가일자별 추이 확인)

In [ ]:
def get_relval_history(ticker, db_info, table_name=TABLE_RESULT):
    tk = to_dg_ticker(ticker)
    sql = f"""
        SELECT date, sector, roe_y2, re_mid, g_est,
               pbr_theory, psr_theory, per_theory,
               tp_pbr, tp_psr, tp_per, tp_avg,
               current_price, upside_avg,
               actual_pbr, actual_psr, actual_per
        FROM `{table_name}`
        WHERE ticker = %s
        ORDER BY date DESC
    """
    conn = get_pymysql_conn(db_info)
    try:
        with conn.cursor() as cur:
            cur.execute(sql, (tk,))
            return pd.DataFrame(cur.fetchall())
    finally:
        conn.close()

# 사용 예
print("[예시] 삼성전자 상대가치 평가 이력")
hist = get_relval_history("A005930", db_info)
display(hist)

# 시각화 (이력이 2개 이상 있을 때만)
if len(hist) >= 2:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    fig.suptitle("A005930 — 상대가치 평가 이력", fontsize=12)

    ax = axes[0]
    hist_sorted = hist.sort_values("date")
    ax.plot(hist_sorted["date"], hist_sorted["current_price"],
            marker="o", label="Current Price", color="#95a5a6")
    ax.plot(hist_sorted["date"], hist_sorted["tp_avg"],
            marker="s", label="Target Price (avg)", color="#e74c3c")
    ax.set_title("Price Path"); ax.set_ylabel("원")
    ax.legend(); ax.grid(alpha=0.3)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_:f"{x:,.0f}"))
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right")

    ax = axes[1]
    ax.plot(hist_sorted["date"], hist_sorted["upside_avg"],
            marker="o", color="#2980b9")
    ax.axhline(0, color="black", lw=1)
    ax.axhline(20, color="green", lw=1, ls="--", alpha=0.6)
    ax.axhline(-20, color="red", lw=1, ls="--", alpha=0.6)
    ax.set_title("Upside Path (%)"); ax.grid(alpha=0.3)
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right")

    plt.tight_layout(); plt.show()
